# 15 — 4-Way Regional: Soft-Voting Ensemble of Notebook 11 RoBERTa Seeds

No retraining. Loads the three RoBERTa-base checkpoints saved by notebook 11 (seeds 42, 123, 2024), reproduces the same train/val/test split deterministically, runs each model on the test set to get logits, averages the softmax probabilities, and reports the ensemble macro-F1 + accuracy.

Reference per-seed test scores from NB 11 (mean ± std): F1 0.817 ± 0.007, accuracy 0.834 ± 0.005.

Expected ensemble lift: +1.5 to +2.5 points on accuracy. Goal: clear 0.85 accuracy.


In [1]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    balanced_accuracy_score, f1_score, classification_report,
)
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

RANDOM_STATE = 42
SEEDS = [42, 123, 2024]
TEXT_COLUMN = 'text_full_concat_scrubbed_plus'
MAX_LENGTH = 256
EVAL_BATCH_SIZE = 32

NB11_ROOT = 'artifacts/origin_region_4way_scrubbed_plus'
SEED_TO_CHECKPOINT = {
    42:   os.path.join(NB11_ROOT, 'roberta_region4way_seed42',   'checkpoint-1992'),
    123:  os.path.join(NB11_ROOT, 'roberta_region4way_seed123',  'checkpoint-1826'),
    2024: os.path.join(NB11_ROOT, 'roberta_region4way_seed2024', 'checkpoint-1992'),
}

OUTPUT_DIR_ROOT = 'artifacts/origin_region_4way_ensemble'
os.makedirs(OUTPUT_DIR_ROOT, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Checkpoints:')
for s, p in SEED_TO_CHECKPOINT.items():
    exists = '✓' if os.path.isdir(p) else '✗ MISSING'
    print(f'  seed {s}: {p}  {exists}')

COUNTRY_TO_REGION = {
    # --- East Africa ---
    'Ethiopia': 'East Africa',
    'Kenya': 'East Africa',
    'Rwanda': 'East Africa',
    'Burundi': 'East Africa',
    'Tanzania': 'East Africa',
    'Uganda': 'East Africa',
    'DR Congo': 'East Africa',
    'Zambia': 'East Africa',
    'Zimbabwe': 'East Africa',
    'Cameroon': 'East Africa',
    'Malawi': 'East Africa',
    'South Africa': 'East Africa',
    'Yemen': 'East Africa',
    # --- Central America ---
    'Guatemala': 'Central America',
    'Costa Rica': 'Central America',
    'Panama': 'Central America',
    'El Salvador': 'Central America',
    'Honduras': 'Central America',
    'Nicaragua': 'Central America',
    'Mexico': 'Central America',
    'Jamaica': 'Central America',
    'Puerto Rico': 'Central America',
    'Haiti': 'Central America',
    'Dominican Republic': 'Central America',
    # --- South America ---
    'Colombia': 'South America',
    'Peru': 'South America',
    'Brazil': 'South America',
    'Ecuador': 'South America',
    'Bolivia': 'South America',
    'Venezuela': 'South America',
    # --- Asia-Pacific ---
    'Indonesia': 'Asia-Pacific',
    'Taiwan': 'Asia-Pacific',
    'Thailand': 'Asia-Pacific',
    'Papua New Guinea': 'Asia-Pacific',
    'Philippines': 'Asia-Pacific',
    'India': 'Asia-Pacific',
    'Vietnam': 'Asia-Pacific',
    'China': 'Asia-Pacific',
    'Timor-Leste': 'Asia-Pacific',
    'Malaysia': 'Asia-Pacific',
    'Laos': 'Asia-Pacific',
    'Nepal': 'Asia-Pacific',
    'Myanmar': 'Asia-Pacific',
    'Australia': 'Asia-Pacific',
    'United Kingdom': 'Asia-Pacific',
    'United States': 'Asia-Pacific',
}


Device: cuda
Checkpoints:
  seed 42: artifacts/origin_region_4way_scrubbed_plus\roberta_region4way_seed42\checkpoint-1992  ✓
  seed 123: artifacts/origin_region_4way_scrubbed_plus\roberta_region4way_seed123\checkpoint-1826  ✓
  seed 2024: artifacts/origin_region_4way_scrubbed_plus\roberta_region4way_seed2024\checkpoint-1992  ✓


## Scrubbing vocabulary (identical to NB 11)


In [2]:
# Scrubbing vocabulary: countries + region aliases + cultivars + producer context
COUNTRY_TERMS = {
    "Ethiopia": ["ethiopia", "ethiopian"],
    "Colombia": ["colombia", "colombian", "columbian"],
    "Panama": ["panama", "panamanian"],
    "Kenya": ["kenya", "kenyan"],
    "Indonesia": ["indonesia", "indonesian"],
    "Guatemala": ["guatemala", "guatemalan"],
    "Costa Rica": ["costa rica", "costa rican", "costarican"],
    "El Salvador": ["el salvador", "salvadoran", "salvadorean", "salvadorian"],
    "Rwanda": ["rwanda", "rwandan"],
    "Brazil": ["brazil", "brazilian"],
    "Honduras": ["honduras", "honduran"],
    "Peru": ["peru", "peruvian"],
    "Taiwan": ["taiwan", "taiwanese"],
    "Papua New Guinea": ["papua new guinea", "papua new guinean"],
    "Nicaragua": ["nicaragua", "nicaraguan"],
    "Burundi": ["burundi", "burundian"],
    "Thailand": ["thailand", "thai"],
    "India": ["india", "indian"],
    "Mexico": ["mexico", "mexican"],
    "Tanzania": ["tanzania", "tanzanian"],
    "Yemen": ["yemen", "yemeni"],
    "Ecuador": ["ecuador", "ecuadorian"],
    "Bolivia": ["bolivia", "bolivian"],
    "Jamaica": ["jamaica", "jamaican"],
    "Uganda": ["uganda", "ugandan"],
    "Dominican Republic": ["dominican republic", "dominican"],
    "Zambia": ["zambia", "zambian"],
    "China": ["china", "chinese"],
    "Vietnam": ["vietnam", "vietnamese"],
    "Philippines": ["philippines", "philippine", "filipino"],
    "Malaysia": ["malaysia", "malaysian"],
    "Laos": ["laos", "laotian"],
    "Zimbabwe": ["zimbabwe", "zimbabwean"],
    "Haiti": ["haiti", "haitian"],
    "Puerto Rico": ["puerto rico", "puerto rican", "puertorican"],
    "Nepal": ["nepal", "nepalese", "nepali"],
    "Myanmar": ["myanmar", "burmese"],
    "Cameroon": ["cameroon", "cameroonian"],
    "Australia": ["australia", "australian"],
    "South Africa": ["south africa", "south african"],
    "Malawi": ["malawi", "malawian"],
    "Venezuela": ["venezuela", "venezuelan", "merida state", "mocoties valley"],
    "Timor-Leste": ["east timor", "timor leste", "timorese"],
    "DR Congo": ["democratic republic of the congo", "dr congo", "congo", "congolese"],
    "United Kingdom": ["united kingdom", "british", "pitcairn island", "saint helena", "st helena", "sandy bay valley"],
    "United States": [
        "usa", "united states", "american",
        "hawaii", "hawai'i", "hawaiian", "hawaii island",
        "big island", "kona", "puna district", "holualoa", "oahu", "maui", "kauai",
        "ka u", "ka'u", "kau",
    ],
}

REGION_ALIASES = {
    "Ethiopia": ["yirgacheffe", "sidamo", "sidama", "guji", "gedeb", "gedeo", "kochere", "hambela",
                 "shakiso", "jimma", "limu", "oromia", "harrar", "kaffa", "bench maji", "bench-maji",
                 "arbegona", "bensa"],
    "Colombia": ["huila", "cauca", "narino", "tolima", "quindio", "caldas", "risaralda", "antioquia",
                 "cundinamarca", "santander", "pitalito", "acevedo", "planadas", "gaitania", "piendamo",
                 "caicedonia", "san agustin", "armenia"],
    "Panama": ["boquete", "chiriqui", "volcan", "jaramillo", "alto quiel", "paso ancho",
               "piedra candela", "canas verdes", "silla del pando", "renacimiento"],
    "Kenya": ["nyeri", "kirinyaga", "kiambu", "embu", "muranga", "murang'a", "thika", "ruiru",
              "mathira", "meru", "nakuru", "gichugu", "karatina"],
    "Indonesia": ["sumatra", "aceh", "gayo", "lintong", "mandheling", "sidikalang", "toraja",
                  "sulawesi", "java", "bali", "kintamani", "flores", "kerinci"],
    "Guatemala": ["huehuetenango", "antigua", "acatenango", "fraijanes", "coban", "atitlan",
                  "chimaltenango", "quiche", "solola", "palencia", "sacatepequez", "san marcos",
                  "hoja blanca", "cuilco"],
    "Costa Rica": ["tarrazu", "central valley", "west valley", "tres rios", "naranjo", "dota",
                   "poas", "alajuela", "brunca", "turrialba", "coto brus", "chirripo"],
    "El Salvador": ["ahuachapan", "chalatenango", "apaneca", "ilamatepec", "santa ana",
                    "el boqueron", "quetzaltepec", "juayua", "ataco"],
    "Rwanda": ["gakenke", "nyamasheke", "karongi", "huye", "nyamagabe", "rulindo", "gikongoro",
               "lake kivu"],
    "Brazil": ["cerrado", "mogiana", "minas gerais", "mantiqueira", "sul de minas",
               "chapada diamantina", "carmo de minas"],
    "Honduras": ["marcala", "copan", "ocotepeque", "comayagua", "intibuca", "santa barbara",
                 "el paraiso", "capucas"],
    "Peru": ["cajamarca", "jaen", "san ignacio", "chanchamayo", "cusco", "villa rica", "oxapampa",
             "junin", "puno"],
    "Taiwan": ["alishan", "yunlin", "chiayi", "chia yi", "nantou", "taichung", "pingtung"],
    "Papua New Guinea": ["wahgi valley", "western highlands", "eastern highlands", "jiwaka",
                         "kainantu", "okapa"],
    "Nicaragua": ["jinotega", "matagalpa", "nueva segovia", "madriz", "ocotal", "dipilto"],
    "Burundi": ["kayanza", "ngozi", "muramvya", "muyinga", "bururi"],
    "Thailand": ["chiang rai", "doi chang", "doi pangkhon", "chiang mai", "nan province"],
    "India": ["coorg", "chikmagalur", "karnataka", "bababudangiri", "nilgiris"],
    "Mexico": ["chiapas", "oaxaca", "veracruz", "coatepec", "pluma hidalgo"],
    "Tanzania": ["mbeya", "ruvuma", "ngorongoro", "arusha", "kilimanjaro", "mbozi"],
    "Yemen": ["haraaz", "haraz", "sanaa", "bani matar", "hayma"],
    "Ecuador": ["loja", "pichincha", "imbabura", "zamora", "chimborazo", "saraguro"],
    "Bolivia": ["caranavi", "yungas", "la paz"],
    "Jamaica": ["blue mountain", "blue mountains"],
    "Uganda": ["rwenzori", "bugisu", "sipi falls"],
    "Vietnam": ["lam dong", "quang tri", "dalat", "cau dat"],
    "Philippines": ["benguet", "bukidnon", "davao"],
    "China": ["yunnan", "baoshan", "puer", "pu'er", "lincong"],
    "Puerto Rico": ["utuado", "yauco", "adjuntas"],
    "Venezuela": ["mocoties valley", "merida state"],
    "Malawi": ["malawi"],
}

REGION_ONLY_TERMS = [
    "central america", "south america", "latin america",
    "east africa", "central africa", "west africa", "africa",
    "asia", "the americas", "americas",
    "central and south america", "south and central america",
    "east and central africa", "various africa growing regions",
]

CULTIVAR_TERMS = [
    "sl28", "sl-28", "sl 28", "sl34", "sl-34", "sl 34",
    "ruiru 11", "ruiru-11", "batian", "k7",
    "gesha", "geisha",
    "pacamara", "pache", "villa sarchi",
    "bourbon", "pink bourbon", "yellow bourbon", "red bourbon",
    "caturra", "catuai", "catuai vermelho", "catuai amarelo",
    "typica", "mundo novo",
    "maragogipe", "maragogype", "maracaturra",
    "castillo", "colombia variety", "variedad colombia", "tabi",
    "heirloom", "ethiopian heirloom",
    "tim tim", "s795", "ateng",
    "catimor", "sarchimor", "icatu", "obata",
]

PRODUCER_CONTEXT_TERMS = [
    "finca", "hacienda", "beneficio",
    "cup of excellence",
    "washing station", "wet mill",
    "best of panama",
]

all_scrub_terms = []
for v in COUNTRY_TERMS.values():    all_scrub_terms.extend(v)
for v in REGION_ALIASES.values():   all_scrub_terms.extend(v)
all_scrub_terms.extend(REGION_ONLY_TERMS)
all_scrub_terms.extend(CULTIVAR_TERMS)
all_scrub_terms.extend(PRODUCER_CONTEXT_TERMS)
all_scrub_terms = sorted(set(all_scrub_terms), key=len, reverse=True)
print(f'Loaded {len(all_scrub_terms)} scrub terms')

SCRUB_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(t) for t in all_scrub_terms) + r')\b',
    flags=re.IGNORECASE,
)

def scrub(text):
    if not isinstance(text, str) or not text:
        return ''
    return SCRUB_PATTERN.sub(' [ORIGIN] ', text)


Loaded 403 scrub terms


## Reproduce NB 11 build + split exactly


In [3]:
def minimal_raw_text(text):
    text = '' if pd.isna(text) else str(text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def normalize_text_keep_brackets(text):
    text = minimal_raw_text(text)
    text = re.sub(r'[^a-z0-9\s\[\]]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

EXTRA_TEXT_COLS = ['Blind Assessment', 'Notes', 'Who Should Drink It', 'Bottom Line']

def concat_and_scrub(row):
    pieces = []
    for col in EXTRA_TEXT_COLS:
        v = row[col]
        if pd.notna(v) and str(v).strip():
            raw = minimal_raw_text(v)
            scrubbed = scrub(raw)
            cleaned = normalize_text_keep_brackets(scrubbed)
            pieces.append(cleaned)
    return ' '.join(p for p in pieces if p)

df = pd.read_csv('Data/final_coffee_reviews.csv')
df['text_full_concat_scrubbed_plus'] = df.apply(concat_and_scrub, axis=1)
df['text_raw_minimal'] = df['Blind Assessment'].fillna('').map(minimal_raw_text)
df['origin_country'] = df['Country'].astype('string').str.strip().replace('', pd.NA)
df['origin_region']  = df['origin_country'].map(COUNTRY_TO_REGION)

work = df[
    (df['text_raw_minimal'].str.len() >= 30)
    & (df['origin_country'].notna())
    & (df['origin_region'].notna())
].copy().reset_index(drop=True)

# Same split as NB 11 (uses y=origin_region for stratification with the same RANDOM_STATE)
y = work['origin_region']
X = work[TEXT_COLUMN].tolist()

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.15/0.85, stratify=y_tmp, random_state=RANDOM_STATE)

label_names = sorted(y.unique().tolist())
label2id = {n: i for i, n in enumerate(label_names)}
id2label = {i: n for i, n in enumerate(label_names)}
test_labels = np.array([label2id[v] for v in y_test])

print(f'Test rows: {len(X_test)} | Labels: {label_names}')
print('Test class counts:', dict(zip(label_names, np.bincount(test_labels, minlength=len(label_names)))))


Test rows: 1138 | Labels: ['Asia-Pacific', 'Central America', 'East Africa', 'South America']
Test class counts: {'Asia-Pacific': 160, 'Central America': 290, 'East Africa': 471, 'South America': 217}


## Load each checkpoint, predict on test, soft-vote


In [4]:
@torch.no_grad()
def get_test_logits(checkpoint_dir):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir).to(DEVICE).eval()

    all_logits = []
    for i in range(0, len(X_test), EVAL_BATCH_SIZE):
        batch_texts = X_test[i:i + EVAL_BATCH_SIZE]
        enc = tokenizer(batch_texts, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt').to(DEVICE)
        out = model(**enc)
        all_logits.append(out.logits.detach().float().cpu().numpy())

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(all_logits, axis=0)

per_seed_logits = {}
per_seed_metrics = {}
for s, ckpt in SEED_TO_CHECKPOINT.items():
    print(f'\n--- seed {s} from {ckpt} ---')
    logits = get_test_logits(ckpt)
    per_seed_logits[s] = logits
    preds = logits.argmax(axis=1)
    f1   = f1_score(test_labels, preds, average='macro', zero_division=0)
    bal  = balanced_accuracy_score(test_labels, preds)
    acc  = accuracy_score(test_labels, preds)
    per_seed_metrics[s] = {'test_f1_macro': float(f1), 'test_bal_acc': float(bal), 'test_accuracy': float(acc)}
    print(f'  F1 macro: {f1:.4f}  bal-acc: {bal:.4f}  accuracy: {acc:.4f}')

# Ensemble: average softmax probs (more stable than averaging raw logits)
def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

probs_stack = np.stack([softmax(per_seed_logits[s]) for s in SEEDS], axis=0)
ens_probs = probs_stack.mean(axis=0)
ens_preds = ens_probs.argmax(axis=1)

ens_f1   = f1_score(test_labels, ens_preds, average='macro', zero_division=0)
ens_bal  = balanced_accuracy_score(test_labels, ens_preds)
ens_acc  = accuracy_score(test_labels, ens_preds)
print('\n=== Ensemble (3-seed soft-vote) ===')
print(f'  F1 macro: {ens_f1:.4f}  bal-acc: {ens_bal:.4f}  accuracy: {ens_acc:.4f}')



--- seed 42 from artifacts/origin_region_4way_scrubbed_plus\roberta_region4way_seed42\checkpoint-1992 ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  F1 macro: 0.8092  bal-acc: 0.8092  accuracy: 0.8269

--- seed 123 from artifacts/origin_region_4way_scrubbed_plus\roberta_region4way_seed123\checkpoint-1826 ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  F1 macro: 0.8203  bal-acc: 0.8171  accuracy: 0.8366

--- seed 2024 from artifacts/origin_region_4way_scrubbed_plus\roberta_region4way_seed2024\checkpoint-1992 ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  F1 macro: 0.8230  bal-acc: 0.8221  accuracy: 0.8409

=== Ensemble (3-seed soft-vote) ===
  F1 macro: 0.8344  bal-acc: 0.8315  accuracy: 0.8497


## Summary + per-class report


In [5]:
print('=' * 60)
print('4-way regional, scrubbed+ full-concat')
print('=' * 60)
print(f"  Per-seed (NB 11 checkpoints):")
for s in SEEDS:
    m = per_seed_metrics[s]
    print(f"    seed {s}: F1={m['test_f1_macro']:.4f}  bal-acc={m['test_bal_acc']:.4f}  acc={m['test_accuracy']:.4f}")
print(f"  Mean across seeds:")
mean_f1   = np.mean([per_seed_metrics[s]['test_f1_macro'] for s in SEEDS])
mean_bal  = np.mean([per_seed_metrics[s]['test_bal_acc'] for s in SEEDS])
mean_acc  = np.mean([per_seed_metrics[s]['test_accuracy'] for s in SEEDS])
print(f'    F1={mean_f1:.4f}  bal-acc={mean_bal:.4f}  acc={mean_acc:.4f}')
print(f"  Ensemble (3-seed soft-vote):")
print(f'    F1={ens_f1:.4f}  bal-acc={ens_bal:.4f}  acc={ens_acc:.4f}')
print(f"  Ensemble lift over per-seed mean:")
print(f'    ΔF1={ens_f1 - mean_f1:+.4f}  Δbal-acc={ens_bal - mean_bal:+.4f}  Δacc={ens_acc - mean_acc:+.4f}')
print()
print('Per-class on test (ensemble):')
print(classification_report(test_labels, ens_preds, target_names=label_names, digits=3, zero_division=0))


4-way regional, scrubbed+ full-concat
  Per-seed (NB 11 checkpoints):
    seed 42: F1=0.8092  bal-acc=0.8092  acc=0.8269
    seed 123: F1=0.8203  bal-acc=0.8171  acc=0.8366
    seed 2024: F1=0.8230  bal-acc=0.8221  acc=0.8409
  Mean across seeds:
    F1=0.8175  bal-acc=0.8161  acc=0.8348
  Ensemble (3-seed soft-vote):
    F1=0.8344  bal-acc=0.8315  acc=0.8497
  Ensemble lift over per-seed mean:
    ΔF1=+0.0169  Δbal-acc=+0.0154  Δacc=+0.0149

Per-class on test (ensemble):
                 precision    recall  f1-score   support

   Asia-Pacific      0.900     0.900     0.900       160
Central America      0.827     0.790     0.808       290
    East Africa      0.899     0.941     0.919       471
  South America      0.726     0.696     0.711       217

       accuracy                          0.850      1138
      macro avg      0.838     0.832     0.834      1138
   weighted avg      0.848     0.850     0.848      1138



## Save results


In [6]:
out = {
    'notebook': '15_Origin_Regional_4Way_Ensemble',
    'task': '4-way regional, soft-vote ensemble of 3 NB 11 RoBERTa seeds',
    'seeds': SEEDS,
    'checkpoints': SEED_TO_CHECKPOINT,
    'per_seed_metrics': per_seed_metrics,
    'per_seed_mean': {
        'test_f1_macro': float(np.mean([per_seed_metrics[s]['test_f1_macro'] for s in SEEDS])),
        'test_bal_acc':  float(np.mean([per_seed_metrics[s]['test_bal_acc']  for s in SEEDS])),
        'test_accuracy': float(np.mean([per_seed_metrics[s]['test_accuracy'] for s in SEEDS])),
    },
    'ensemble': {
        'test_f1_macro': float(ens_f1),
        'test_bal_acc':  float(ens_bal),
        'test_accuracy': float(ens_acc),
    },
    'ensemble_lift_vs_mean': {
        'd_f1_macro': float(ens_f1 - np.mean([per_seed_metrics[s]['test_f1_macro'] for s in SEEDS])),
        'd_bal_acc':  float(ens_bal - np.mean([per_seed_metrics[s]['test_bal_acc']  for s in SEEDS])),
        'd_accuracy': float(ens_acc - np.mean([per_seed_metrics[s]['test_accuracy'] for s in SEEDS])),
    },
}
out_path = os.path.join(OUTPUT_DIR_ROOT, 'results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=float)
print('Saved:', out_path)


Saved: artifacts/origin_region_4way_ensemble\results.json
